# TirraMind Phase 50 — Eval Only Notebook

This notebook is intentionally separate from training.

- No retraining
- Load trained artifacts (`epoch_090.pt` + `gnn_model_phase50.pt`)
- Run walk-forward backtest/evaluation only


In [ ]:
import json
from pathlib import Path

EVAL_CONFIG = {
    "phase": "50-eval",
    "source_version": 55,
    "target_epoch": 90,
    "require_artifact_min_bytes": 1_000_000,
    "smoke": False,
}

print(json.dumps(EVAL_CONFIG, indent=2))


In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

pip("torch-geometric==2.7.0")
pip("rich", "tqdm", "scipy>=1.14.0")

import torch
import torch_geometric
import scipy
print("torch", torch.__version__)
print("pyg", torch_geometric.__version__)
print("scipy", scipy.__version__)


In [ ]:
import os
import shutil
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Mounted input datasets:")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(" ", p.name)

def find_code_root(root="/kaggle/input"):
    for dirpath, dirs, _ in os.walk(root):
        if {"agent", "scripts"}.issubset(set(dirs)):
            return Path(dirpath)
    return None

def find_data_root(root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if "pipeline.db" in files:
            return Path(dirpath)
    return None

_code_root = find_code_root()
if _code_root is None:
    raise RuntimeError(
        "CODE NOT FOUND under /kaggle/input. Attach deeperisbetter/tirramind-code."
    )

for name in ("agent", "scripts"):
    dst = WORK_DIR / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(_code_root / name, dst)

_data_root = find_data_root()
if _data_root is None:
    raise RuntimeError(
        "DATA NOT FOUND under /kaggle/input. Attach deeperisbetter/tirramind-data."
    )

PIPELINE_DIR = WORK_DIR / ".tirra_pipeline"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DB_DST = PIPELINE_DIR / "pipeline.db"
shutil.copy2(_data_root / "pipeline.db", PIPELINE_DB_DST)

sys.path.insert(0, str(WORK_DIR))
print(f"Code from {_code_root}")
print(f"Data from {_data_root}")
print(f"pipeline.db -> {PIPELINE_DB_DST}")


In [ ]:
from pathlib import Path

def find_best_file(filename: str, min_bytes: int) -> Path:
    candidates = []
    for p in Path("/kaggle/input").rglob(filename):
        try:
            size = p.stat().st_size
        except OSError:
            continue
        if size >= min_bytes:
            candidates.append((size, p))
    if not candidates:
        raise FileNotFoundError(f"No valid {filename} >= {min_bytes} bytes under /kaggle/input")
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

model_src = find_best_file("gnn_model_phase50.pt", EVAL_CONFIG["require_artifact_min_bytes"])
epoch_src = find_best_file(f"epoch_{EVAL_CONFIG['target_epoch']:03d}.pt", EVAL_CONFIG["require_artifact_min_bytes"])

print("model:", model_src, model_src.stat().st_size)
print("epoch:", epoch_src, epoch_src.stat().st_size)


In [ ]:
import subprocess
import sys
from pathlib import Path

out_json = Path("/kaggle/working/ic_results_eval_phase50.json")

diag_out = Path("/kaggle/working/eval_diagnostics.json")
diag_cmd = [
    sys.executable, "scripts/gnn_eval_diagnostics.py",
    "--model-path", str(model_src),
    "--weights-from-epoch", str(epoch_src),
    "--db-path", str(PIPELINE_DB_DST),
    "--out", str(diag_out),
]
if EVAL_CONFIG.get("smoke"):
    diag_cmd.append("--smoke")
print("Running:", " ".join(diag_cmd))
subprocess.run(diag_cmd, cwd=str(WORK_DIR), check=True)

cmd = [
    sys.executable,
    "scripts/phase40_gnn_backtest.py",
    "--model-path", str(model_src),
    "--weights-from-epoch", str(epoch_src),
    "--db-path", str(PIPELINE_DB_DST),
    "--out", str(out_json),
]
if EVAL_CONFIG.get("smoke"):
    cmd.append("--smoke")

print("Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=str(WORK_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Backtest failed with exit code {rc}")
print("Done. Results:", out_json)


In [ ]:
import json
from pathlib import Path

out_json = Path("/kaggle/working/ic_results_eval_phase50.json")
res = json.loads(out_json.read_text())

print("Model:", res.get("model_path"))
print("Weights:", res.get("weights_from_epoch"))

for name, m in res.get("strategies", {}).items():
    print(f"{name:18s} mean_ic={m['mean_ic']:+.4f}  t={m['t_stat']:+.2f}  n={m['n_folds']}")

best = res.get("best", {})
if best:
    print("Best strategy:", best.get("strategy"), "mean_ic", round(best.get("mean_ic", 0.0), 4))
